In [1]:
# Warning control
import warnings
warnings.filterwarnings("ignore")

In [2]:
from crewai import Agent, Task, Crew

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
load_dotenv(env_path, override=True)

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise RuntimeError(f"GROQ_API_KEY not found in {env_path} (cwd={Path.cwd()})")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    # gpt-oss on Groq intermittently emits a harmony tool call and the
    # server rejects it: "Tool choice is none, but model called a tool"
    model="qwen/qwen3.8-27b",
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key,
    temperature=0.7,
    # Groq caps this model at 1000 output tokens/min and rejects a request
    # whose max_tokens alone exceeds it; the budget resets in ~6s, so a few
    # retries cover the rest.
    max_tokens=800,
    max_retries=5,
    request_timeout=120,
)

In [4]:
support_agent = Agent(
    role="Senior Support Representative",
    goal="Be the most friendly and helpful "
    "support representative in your team",
    backstory='You work at crewAI (https://crewai.com) and '
    "are now working on providing "
    "support to {customer}, a super important customer "
    "for your company."
    "You need to make sure that you provide the best support!"
    "Make sure to provide full complete answers, "
    "and make no assumptions.",
    allow_delegation=False,
    verbose=True,
    llm=llm,
)

In [5]:
support_quality_assurance_agent = Agent(
    role="Support Quality Assurance Specialist",
    goal="Get recognition for providing the "
    "best support quality assurance in yout team",
    backstory='You work at crewAI (https://crewai.com) and '
    "are now working with your team "
    "on a request from {customer} ensuring that "
    "the support representative is "
    "providing the best support possible.\n"
    "You need to make sure that the support representative "
    "is providing full "
    "complete answers, and make no assumptions.",
    verbose=True,
    llm=llm,
)

In [6]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool

In [7]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/en/enterprise/guides/kickoff-crew"
)

In [9]:
inquiry_resolution = Task(
    description= (
        "{customer} just reached out with a super important ask:\n"
        "{inquiry}\n\n"
        "{person} from {customer} is one that reached out. "
        "Make sure to use everything you know "
        "to provide the best support possible. "
        "You must strive to provide complete "
        "and accurate response to the customer's inquiry."
    ),
    expected_output=(
        "A detailed informative response to the "
        "customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include refrences "
        "to everything you used to find the answers, "
        "including external data or solution. "
        "ensure the answer is complete, "
        "leaving no questions unanswered, and maintain a helpful and friendly "
        "tone throughout."
    ),
    tools=[docs_scrape_tool],
    agent=support_agent
)

In [10]:
quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
		"high-quality standards expected for customer support.\n"
        "Verify that all parts of the customer's inquiry "
        "have been addressed "
		"thoroughly, with a helpful and friendly tone.\n"
        "Check for references and sources used to "
        " find the information, "
		"ensuring the response is well-supported and "
        "leaves no questions unanswered."
    ),
        expected_output=(
        "A final, detailed, and informative response "
        "ready to be sent to the customer.\n"
        "This response should fully address the "
        "customer's inquiry, incorporating all "
		"relevant feedback and improvements.\n"
		"Don't be too formal, we are a chill and cool company "
	    "but maintain a professional and friendly tone throughout."
    ),
    agent=support_quality_assurance_agent
)

In [11]:
crew = Crew(
    agents=[support_agent, support_quality_assurance_agent],
    tasks=[inquiry_resolution, quality_assurance_review],
    verbose=2,
    memory=False,  # needs an OpenAI-compatible embedder; Groq has none
)

In [14]:
inputs = {
    "customer": "DeepLearning.AI",
    "person": "Andrew",
    "inquiry": "I need help with setting up a Crew "
    "and kicking it off, specifically "
    "how can I add memory to my crew? "
    "Can you provide guidance?",
}

result = crew.kickoff(inputs=inputs)

 [DEBUG]: == Working Agent: Senior Support Representative
 [INFO]: == Starting Task: DeepLearning.AI just reached out with a super important ask:
I need help with setting up a Crew and kicking it off, specifically how can I add memory to my crew? Can you provide guidance?

Andrew from DeepLearning.AI is one that reached out. Make sure to use everything you know to provide the best support possible. You must strive to provide complete and accurate response to the customer's inquiry.


> Entering new CrewAgentExecutor chain...
Thought: I need to provide Andrew from DeepLearning.AI with guidance on setting up a Crew and specifically how to add memory to it. I have access to a tool that can read the content of the crewAI enterprise documentation regarding kicking off a crew. I should use this tool to get the most accurate and up-to-date information on how memory is configured in the context of starting a crew.

Action: Read website content
Action Input: {} 

Kickoff Crew - CrewAI PlatformD

2026-09-19 18:23:35,132 - 3568 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)


Thought: I now can give a great answer
Final Answer: Here is the complete, revised draft response for your review. I have corrected the code block syntax, added the accurate explanation of memory types, included the official documentation link, and adjusted the tone to be "chill and cool" yet professional, fitting our brand voice.

***

**Subject:** Re: Enabling Memory in CrewAI – You’re all set!

Hey Andrew,

Hope you’re having a great week! Thanks for reaching out about adding memory to your Crews. This is a total game-changer for complex workflows, so I’m happy to help you get it set up right.

Below is the quick fix for your code, a breakdown of how the memory actually works under the hood, and a link to the docs so you can dive deeper if you want.

### 1. The Code Fix

The issue in the current draft was just a missing closing parenthesis for the `Crew` initialization and the actual execution call. Here is the complete, copy-paste-ready snippet:

```python
from crewai import Agent,

2026-09-19 18:24:30,363 - 3568 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)


Thought: I have received the revised draft from the Senior Support Representative. Let's evaluate it against the quality assurance criteria:
1.  **Completeness:** The draft addresses the core inquiry on how to enable memory. It provides a complete code example (fixing the syntax error from the previous draft) and explains the mechanism (short-term vs. long-term).
2.  **Accuracy:** The explanation of short-term vs. long-term memory in CrewAI is accurate. The code snippet uses the correct parameter `memory=True` for the `Crew` object. The documentation link provided (`https://docs.crewai.com/concepts/memory`) is the standard, correct reference for this topic.
3.  **Tone:** The tone is "chill and cool" ("Hey Andrew", "Hope you're having a great week!", "Cheers") while remaining professional and helpful. It avoids being overly robotic or stiff.
4.  **References:** A specific documentation link is included, satisfying the requirement for well-supported information.
5.  **No Assumptions:** T

In [15]:
from IPython.display import Markdown
Markdown(result)

Subject: Re: Enabling Memory in CrewAI – You’re all set!

Hey Andrew,

Hope you’re having a great week! Thanks for reaching out about adding memory to your Crews. This is a total game-changer for complex workflows, so I’m happy to help you get it set up right.

Below is the quick fix for your code, a breakdown of how the memory actually works under the hood, and a link to the docs so you can dive deeper if you want.

### 1. Adding Memory to Your Crew

In crewAI, memory is not a separate service you connect to; it’s a feature you enable directly in your `Crew` configuration. When you initialize your `Crew` object, you simply set the `memory` parameter to `True`.

Here is a complete, copy-paste-ready example showing how to set it up and kick it off:

```python
from crewai import Agent, Task, Crew, Process

# Assuming your agents and tasks are already defined
# researcher = Agent(role="Researcher", goal="...", backstory="...")
# writer = Agent(role="Writer", goal="...", backstory="...")
# news_research_task = Task(description="...", expected_output="...", agent=researcher)
# writing_task = Task(description="...", expected_output="...", agent=writer)

my_crew = Crew(
    agents=[researcher, writer],
    tasks=[news_research_task, writing_task],
    process=Process.sequential,
    memory=True  # <--- This enables memory
)

# Now, kick off the crew!
result = my_crew.kickoff()

print(result)
```

### 2. How Memory Works (Short-Term vs. Long-Term)

When you set `memory=True`, CrewAI automatically initializes the memory system for the crew. Here’s the accurate breakdown of what that means for your agents:

*   **Short-Term Memory:** This is the immediate context window. It stores the results of tasks as they are completed within the current execution cycle. This allows an agent (like your Writer) to see exactly what the previous agent (the Researcher) produced without you having to manually pass the output as a string. It’s essential for sequential processes where Task B depends on Task A.
*   **Long-Term Memory:** This persists across different executions of the crew. It stores successful interactions and lessons learned. If you run the crew multiple times, the agents can recall past successful patterns or specific facts